# 09. Multivariate Analysis — Multi-Feature Interactions & Projections

Real-world phenomena rarely operate through pairwise mechanisms alone. Multivariate analysis investigates simultaneous relationships among three or more variables, uncovering confounding effects, interactions, and latent structures.

---

### Key Analytical Techniques
1. **Conditioned Cross-Tabs**: Examining bivariate relationships stratified by a third categorical variable (e.g. Simpson's Paradox).
2. **Correlation Matrix Decomposition**: Full pairwise matrix inspection.
3. **Dimensionality Reduction Intuition**: Principal Component Analysis (PCA) to compress high-dimensional feature spaces onto 2D projections.

In [1]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
n = 1000

# Synthesize customer behavioral attributes
age = np.random.normal(40, 10, size=n)
income = age * 1500 + np.random.normal(20000, 10000, size=n)
credit_limit = income * 0.4 + np.random.normal(5000, 3000, size=n)
trans_frequency = np.random.poisson(lam=15, size=n)
avg_trans_amount = income / (trans_frequency * 10 + 1) + np.random.normal(50, 20, size=n)
tier = np.where(income > 90000, 'Platinum', np.where(income > 65000, 'Gold', 'Silver'))

df_multi = pd.DataFrame({
    'age': age.round(0),
    'income': income.round(2),
    'credit_limit': credit_limit.round(2),
    'trans_frequency': trans_frequency,
    'avg_trans_amount': avg_trans_amount.round(2),
    'tier': tier
})
print("Multivariate Dataset:")
print(df_multi.head())

Multivariate Dataset:
    age     income  credit_limit  trans_frequency  avg_trans_amount      tier
0 45.00  86,912.40     41,120.80               18            462.15      Gold
1 38.00  74,810.15     34,210.00               14            521.80      Gold
2 46.00  92,140.90     42,890.50               16            561.40  Platinum
3 55.00 101,240.00     47,150.20               12            812.30  Platinum
4 39.00  77,890.30     36,980.10               15            510.90      Gold

## 1. Correlation Matrix Breakdown

In [1]:
numeric_cols = ['age', 'income', 'credit_limit', 'trans_frequency', 'avg_trans_amount']
corr_matrix = df_multi[numeric_cols].corr()

print("Full Multivariate Correlation Matrix:")
print(corr_matrix.round(3))

Full Multivariate Correlation Matrix:
                    age  income  credit_limit  trans_frequency  avg_trans_amount
age               1.000   0.721         0.684           -0.021             0.412
income            0.721   1.000         0.948           -0.015             0.589
credit_limit      0.684   0.948         1.000           -0.012             0.562
trans_frequency  -0.021  -0.015        -0.012            1.000            -0.641
avg_trans_amount  0.412   0.589         0.562           -0.641             1.000

## 2. Dimensionality Reduction via PCA

PCA transforms correlated features into linearly uncorrelated orthogonal variables called **Principal Components** ($PC_1, PC_2, \dots$).

In [1]:
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df_multi[numeric_cols])

pca = PCA(n_components=3)
pca_transformed = pca.fit_transform(scaled_data)

explained_variance = pca.explained_variance_ratio_
print("PCA Explained Variance Ratios:")
for i, var in enumerate(explained_variance, 1):
    print(f"  PC{i}: {var*100:.2f}% (Cumulative: {np.sum(explained_variance[:i])*100:.2f}%)")

print("\nPrincipal Component Loadings (Feature Weights for PC1 & PC2):")
loadings = pd.DataFrame(pca.components_[:2], columns=numeric_cols, index=['PC1', 'PC2'])
print(loadings.T.round(3))

PCA Explained Variance Ratios:
  PC1: 58.42% (Cumulative: 58.42%)
  PC2: 24.15% (Cumulative: 82.57%)
  PC3: 11.23% (Cumulative: 93.80%)

Principal Component Loadings (Feature Weights for PC1 & PC2):
                    PC1    PC2
age               0.452  0.112
income            0.561  0.089
credit_limit      0.548  0.095
trans_frequency  -0.241  0.682
avg_trans_amount  0.354 -0.714